In [ ]:
import concurrent.futures
import csv
import hashlib
import json
import os
import re
from pathlib import Path
from itertools import combinations
from typing import Any, Dict, Iterable, List, Optional, Tuple

from SPARQLWrapper import JSON, SPARQLWrapper

try:
    from rdflib.plugins.sparql.parser import parseQuery
except Exception:
    parseQuery = None


DEFAULT_RESULTS_JSONL = "batch_results_prediction-model-2026-05-01T20_10_27.865167Z_predictions.jsonl"
DEFAULT_OUTPUT_JSONL = "sparql_evaluation_results.jsonl"
DEFAULT_OUTPUT_CSV = "sparql_evaluation_results.csv"
DEFAULT_SUMMARY_JSONL = "sparql_evaluation_summary.jsonl"
DEFAULT_SUMMARY_CSV = "sparql_evaluation_summary.csv"
FUSEKI_BASE_URL = "<FUSEKI_BASE_URL>"  # e.g. "http://localhost:3030"
DEFAULT_ENDPOINT = f"{FUSEKI_BASE_URL}/currkg/sparql"

# Override any graph-specific endpoints here if a graph lives in a different Fuseki dataset.
ENDPOINT_BY_GRAPH = {
    "default": DEFAULT_ENDPOINT,
    "kwg": f"{FUSEKI_BASE_URL}/kwg/sparql",
    "kwg_lite": f"{FUSEKI_BASE_URL}/kwg_lite/sparql",
    "currkg": f"{FUSEKI_BASE_URL}/currkg/sparql",
    "enslaved": f"{FUSEKI_BASE_URL}/enslaved/sparql",
    "enslaved_wiki": f"{FUSEKI_BASE_URL}/enslaved_wiki/sparql",
    "gbo": f"{FUSEKI_BASE_URL}/gbo/sparql",
    "gmo": f"{FUSEKI_BASE_URL}/gmo/sparql",
    "core_scholar_rich": f"{FUSEKI_BASE_URL}/core_scholar_rich/sparql",
    "core_scholar_shallow": f"{FUSEKI_BASE_URL}/core_scholar_shallow/sparql",
}

SCHEMA_PREFIX_FILES = {
    "kwg": "schemas/kwg/schema.ttl",
    "kwg_lite": "schemas/kwg_lite/schema.ttl",
    "currkg": "schemas/currkg/schema.ttl",
    "enslaved": "schemas/enslaved/schema.ttl",
    "enslaved_wiki": "schemas/enslaved_wiki/schema.ttl",
    "gbo": "schemas/gbo/schema.ttl",
    "gmo": "schemas/gmo/schema.ttl",
    "core_scholar_rich": "schemas/core_scholar_rich/schema.ttl",
    "core_scholar_shallow": "schemas/core_scholar_shallow/schema.ttl",
}

CQ_FILES = {
    "kwg": "cqs/kwg.txt",
    "kwg_lite": "cqs/kwg_lite.txt",
    "currkg": "cqs/currkg.txt",
    "enslaved": "cqs/enslaved.txt",
    "enslaved_wiki": "cqs/enslaved_wiki.txt",
    "gbo": "cqs/gbo.txt",
    "gmo": "cqs/gmo.txt",
    "core_scholar_rich": "cqs/core_scholar_rich.txt",
    "core_scholar_shallow": "cqs/core_scholar_shallow.txt",
}


def normalize_endpoint(endpoint: str) -> str:
    if not endpoint:
        return endpoint
    if endpoint.startswith(("http://", "https://")):
        return endpoint
    return "http://" + endpoint


def _run(endpoint: str, query: str, headers: Optional[Dict[str, str]] = None, timeout: int = 600):
    """Execute SPARQL and return (bindings, raw_json)."""
    sparql = SPARQLWrapper(normalize_endpoint(endpoint))
    sparql.setTimeout(timeout)
    sparql.setReturnFormat(JSON)
    sparql.setQuery(query)
    if headers:
        for key, value in headers.items():
            sparql.addCustomHttpHeader(key, value)
    raw = sparql.query().convert()
    bindings = raw.get("results", {}).get("bindings", [])
    return bindings, raw


def _normalize_binding(binding: Dict[str, Dict[str, str]]) -> Dict[str, str]:
    out = {}
    for key, value in binding.items():
        normalized = value.get("value", "")
        if "xml:lang" in value:
            normalized += f"@{value['xml:lang']}"
        if "datatype" in value:
            normalized += f"^^<{value['datatype']}>"
        out[key] = normalized
    return out


def _result_checksum(bindings: List[Dict[str, Dict[str, str]]], raw: Optional[Dict[str, Any]] = None) -> str:
    if raw is not None and "boolean" in raw:
        payload = {"boolean": raw.get("boolean")}
    else:
        rows = [tuple(sorted(_normalize_binding(binding).items())) for binding in bindings]
        rows.sort()
        payload = rows
    return hashlib.sha256(repr(payload).encode("utf-8")).hexdigest()


def _extract_where(query: str) -> Optional[str]:
    """Extract the WHERE/body group with simple brace matching."""
    if not isinstance(query, str):
        return None

    where_match = re.search(r"\bWHERE\b", query, flags=re.IGNORECASE)
    search_start = where_match.end() if where_match else 0
    start = query.find("{", search_start)
    if start < 0:
        return None

    depth = 0
    for idx in range(start, len(query)):
        char = query[idx]
        if char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return query[start + 1:idx]
            if depth < 0:
                return None
    return None


def _extract_prefix_block(query: str) -> str:
    lines = re.findall(r"(?im)^\s*(?:PREFIX|BASE)\s+[^\r\n]+", query or "")
    return "\n".join(lines) + ("\n" if lines else "")


In [ ]:
_prefix_cache: Dict[str, Dict[str, str]] = {}


def load_prefixes_from_ttl(ttl_path: str) -> Dict[str, str]:
    """Read @prefix declarations from a local TTL schema file."""
    if not ttl_path:
        return {}
    if ttl_path in _prefix_cache:
        return _prefix_cache[ttl_path]

    prefixes = {}
    try:
        with open(ttl_path, "r", encoding="utf-8") as handle:
            for line in handle:
                match = re.match(r"\s*@prefix\s+([A-Za-z][A-Za-z0-9_-]*):\s*<([^>]+)>\s*\.", line)
                if match:
                    prefixes[match.group(1)] = match.group(2)
    except FileNotFoundError:
        print(f"Prefix file not found: {ttl_path}; continuing without schema prefixes")

    _prefix_cache[ttl_path] = prefixes
    return prefixes


def extract_sparql(text: str) -> str:
    """Extract a SPARQL query from fenced or plain LLM output."""
    if not isinstance(text, str):
        return ""

    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE).strip()
    if "\\n" in text or "\\t" in text or "\\r" in text:
        text = text.replace("\\r", "\r").replace("\\n", "\n").replace("\\t", "\t")

    fenced = re.findall(r"```(?:sparql)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
    if fenced:
        return fenced[0].strip()

    lowered = text.lower()
    starts = [lowered.find(keyword) for keyword in ("prefix", "base", "select", "ask", "construct", "describe")]
    starts = [idx for idx in starts if idx >= 0]
    if starts:
        return text[min(starts):].strip()
    return text.strip()


def align_prefixes_to_schema(query: str, ttl_prefixes: Dict[str, str]) -> str:
    """
    Make local TTL/schema prefixes authoritative for evaluation.

    - If a generated query declares a prefix that also appears in the TTL file,
      replace the generated URI with the TTL URI.
    - If a generated query uses a schema prefix but does not declare it, prepend it.
    - Prefixes not present in the TTL file are left unchanged.
    """
    if not query or not ttl_prefixes:
        return query

    declared = set()

    def replace_prefix(match: re.Match) -> str:
        keyword = match.group(1)
        prefix = match.group(2)
        uri = match.group(3)
        declared.add(prefix)
        if prefix not in ttl_prefixes:
            return match.group(0)
        replacement_uri = ttl_prefixes[prefix]
        if keyword.lower() == "@prefix":
            return f"@prefix {prefix}: <{replacement_uri}> ."
        return f"{keyword} {prefix}: <{replacement_uri}>"

    aligned_query = re.sub(
        r"(?im)^[ \t]*(@prefix|PREFIX)[ \t]+([A-Za-z][A-Za-z0-9_-]*):[ \t]*<([^>]+)>[ \t]*\.?",
        replace_prefix,
        query,
    )

    used = set(re.findall(r"\b([A-Za-z][A-Za-z0-9_-]*):[A-Za-z_][\w.-]*", aligned_query))
    missing = sorted((used - declared).intersection(ttl_prefixes))
    if not missing:
        return aligned_query

    prefix_block = "\n".join(f"PREFIX {prefix}: <{ttl_prefixes[prefix]}>" for prefix in missing)
    return prefix_block + "\n" + aligned_query.lstrip()


# Backward-compatible name used by earlier notebook cells.
def add_missing_prefixes(query: str, ttl_prefixes: Dict[str, str]) -> str:
    return align_prefixes_to_schema(query, ttl_prefixes)

def get_candidate_text(record: Dict[str, Any]) -> str:
    """Return the first model text from a Gemini batch JSONL record."""
    candidates = record.get("response", {}).get("candidates", []) or []
    for candidate in candidates:
        parts = candidate.get("content", {}).get("parts", []) or []
        text = "".join(part.get("text", "") for part in parts if isinstance(part, dict))
        if text.strip():
            return text
    return ""


def get_competency_question(record: Dict[str, Any]) -> str:
    request = record.get("request", {}) or {}
    contents = request.get("contents", []) or []
    text = "\n".join(
        part.get("text", "")
        for content in contents
        for part in (content.get("parts", []) or [])
        if isinstance(part, dict)
    )
    match = re.search(r"competency question:\s*(.*?)(?:\n\n|\nRequirements:|$)", text, flags=re.IGNORECASE | re.DOTALL)
    if match:
        return " ".join(match.group(1).split())
    return ""



def parse_key_metadata(key: str) -> Dict[str, Any]:
    """
    Parse result keys shaped like:
      task-representation-kg-prompt_type-tempX-question text

    Example:
      sparql-nen-core_scholar_rich-0_shot-temp1.0-Which authors...
    """
    parts = (key or "").split("-", 5)
    metadata = {
        "task": "",
        "representation": "",
        "kg": "",
        "prompt_type": "",
        "temperature": None,
        "temperature_label": "",
        "question_from_key": "",
    }
    if len(parts) >= 1:
        metadata["task"] = parts[0]
    if len(parts) >= 2:
        metadata["representation"] = parts[1]
    if len(parts) >= 3:
        metadata["kg"] = parts[2]
    if len(parts) >= 4:
        metadata["prompt_type"] = parts[3]
    if len(parts) >= 5:
        temp_label = parts[4]
        metadata["temperature_label"] = temp_label
        temp_match = re.match(r"temp([+-]?\d+(?:\.\d+)?)$", temp_label)
        if temp_match:
            metadata["temperature"] = float(temp_match.group(1))
    if len(parts) >= 6:
        metadata["question_from_key"] = parts[5]
    return metadata

_cq_complexity_cache: Dict[str, Dict[str, Tuple[int, str]]] = {}
_global_cq_complexity_cache: Optional[Dict[str, Tuple[int, str]]] = None


def _normalize_cq_text(text: str) -> str:
    """Normalize CQ text for matching JSONL keys/prompts to cqs/*.txt lines."""
    text = (text or "").strip().lower()
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return " ".join(text.split())


def _complexity_for_index(index_1based: int) -> str:
    if index_1based <= 5:
        return "simple"
    if index_1based <= 10:
        return "moderate"
    return "complex"


def load_cq_complexity_map(cq_file: str) -> Dict[str, Tuple[int, str]]:
    """Return {normalized_question: (1-based index, complexity)} for one CQ file."""
    if not cq_file:
        return {}
    if cq_file in _cq_complexity_cache:
        return _cq_complexity_cache[cq_file]

    mapping: Dict[str, Tuple[int, str]] = {}
    try:
        with open(cq_file, "r", encoding="utf-8") as handle:
            questions = [line.strip() for line in handle.read().splitlines() if line.strip()]
    except FileNotFoundError:
        questions = []

    for idx, question in enumerate(questions, start=1):
        mapping[_normalize_cq_text(question)] = (idx, _complexity_for_index(idx))
    _cq_complexity_cache[cq_file] = mapping
    return mapping


def load_global_cq_complexity_map() -> Dict[str, Tuple[int, str]]:
    """Fallback CQ lookup across all known single-KG CQ files."""
    global _global_cq_complexity_cache
    if _global_cq_complexity_cache is not None:
        return _global_cq_complexity_cache

    merged: Dict[str, Tuple[int, str]] = {}
    for cq_file in CQ_FILES.values():
        for question, value in load_cq_complexity_map(cq_file).items():
            merged.setdefault(question, value)
    _global_cq_complexity_cache = merged
    return merged


def infer_cq_complexity(question: str, kg: str) -> Tuple[Optional[int], str]:
    """Infer CQ index and simple/moderate/complex bucket from cqs/<kg>.txt."""
    normalized = _normalize_cq_text(question)
    if not normalized:
        return None, "unknown"

    kg_match = load_cq_complexity_map(CQ_FILES.get(kg, "")).get(normalized)
    if kg_match:
        return kg_match

    global_match = load_global_cq_complexity_map().get(normalized)
    if global_match:
        return global_match
    return None, "unknown"


def infer_graph_id(record: Dict[str, Any]) -> str:
    metadata = parse_key_metadata(record.get("key", "") or "")
    if metadata.get("kg") in SCHEMA_PREFIX_FILES:
        return metadata["kg"]

    key = record.get("key", "") or ""
    candidates = sorted(SCHEMA_PREFIX_FILES, key=len, reverse=True)
    for graph_id in candidates:
        if graph_id in key:
            return graph_id
    return "default"


def iter_jsonl(path: str) -> Iterable[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                record = json.loads(line)
                record["_line_number"] = line_number
                yield record


In [ ]:
def syntax_ok(query: str) -> Tuple[bool, str]:
    """
    Return whether the query parses as SPARQL.

    Uses rdflib's SPARQL parser when available, falling back to basic structural checks.
    """
    if not isinstance(query, str) or not query.strip():
        return False, "Empty query"
    if query.count("{") != query.count("}"):
        return False, "Unbalanced braces"
    if not re.search(r"\b(SELECT|ASK|CONSTRUCT|DESCRIBE)\b", query, flags=re.IGNORECASE):
        return False, "No SPARQL query form found"

    if parseQuery is not None:
        try:
            parseQuery(query)
        except Exception as exc:
            return False, str(exc)
    return True, "OK"


# Backward-compatible name from the original notebook.
def syntax_valid(query: str) -> Tuple[bool, str]:
    return syntax_ok(query)


def satisfiable(endpoint: str, query: str, timeout: int = 600) -> Tuple[bool, str]:
    """Return whether the query's main graph pattern has at least one match."""
    ok, message = syntax_ok(query)
    if not ok:
        return False, f"Syntax failed: {message}"

    body = _extract_where(query)
    if not body:
        return False, "Could not extract WHERE/body group"

    ask_query = f"{_extract_prefix_block(query)}ASK WHERE {{ {body} }}"
    try:
        _, raw = _run(endpoint, ask_query, timeout=timeout)
    except Exception as exc:
        return False, str(exc)
    return bool(raw.get("boolean", False)), "OK"


def deterministic(endpoint: str, query: str, runs: int = 3, timeout: int = 600) -> Tuple[bool, str]:
    """Run the same query repeatedly and compare order-insensitive result checksums."""
    ok, message = syntax_ok(query)
    if not ok:
        return False, f"Syntax failed: {message}"

    checksums = []
    try:
        for _ in range(runs):
            bindings, raw = _run(endpoint, query, timeout=timeout)
            checksums.append(_result_checksum(bindings, raw))
    except Exception as exc:
        return False, str(exc)
    return len(set(checksums)) == 1, "OK"


# Compatibility alias for the spelling in the request.
def deteerminsistic(endpoint: str, query: str, runs: int = 3, timeout: int = 600) -> Tuple[bool, str]:
    return deterministic(endpoint, query, runs=runs, timeout=timeout)


def result_shape(endpoint: str, query: str, timeout: int = 600) -> Tuple[int, List[str], str]:
    """Return row count and projected variables for SELECT/ASK JSON responses."""
    try:
        bindings, raw = _run(endpoint, query, timeout=timeout)
    except Exception as exc:
        return 0, [], str(exc)

    if "boolean" in raw:
        return int(bool(raw.get("boolean"))), [], "OK"
    variables = raw.get("head", {}).get("vars", []) or []
    return len(bindings), variables, "OK"


In [ ]:
def evaluate_record(
    record: Dict[str, Any],
    endpoint_by_graph: Optional[Dict[str, str]] = None,
    runs: int = 3,
    timeout: int = 600,
    add_schema_prefixes: bool = True,
) -> Dict[str, Any]:
    """Evaluate one JSONL result record and return a flat result dict."""
    endpoint_by_graph = endpoint_by_graph or ENDPOINT_BY_GRAPH
    key_metadata = parse_key_metadata(record.get("key", "") or "")
    graph_id = key_metadata.get("kg") or infer_graph_id(record)
    endpoint = endpoint_by_graph.get(graph_id) or endpoint_by_graph.get("default") or DEFAULT_ENDPOINT

    raw_text = get_candidate_text(record)
    query = extract_sparql(raw_text)
    if add_schema_prefixes:
        query = add_missing_prefixes(query, load_prefixes_from_ttl(SCHEMA_PREFIX_FILES.get(graph_id, "")))

    syntax_passed, syntax_message = syntax_ok(query)
    satisfiable_passed, satisfiable_message = (False, "Skipped because syntax failed")
    deterministic_passed, deterministic_message = (False, "Skipped because syntax failed")
    rows, variables, result_message = 0, [], "Skipped because syntax failed"

    if syntax_passed:
        satisfiable_passed, satisfiable_message = satisfiable(endpoint, query, timeout=timeout)
        deterministic_passed, deterministic_message = deterministic(endpoint, query, runs=runs, timeout=timeout)
        rows, variables, result_message = result_shape(endpoint, query, timeout=timeout)

    competency_question = get_competency_question(record) or key_metadata.get("question_from_key", "")
    cq_index, cq_complexity = infer_cq_complexity(competency_question, graph_id)
    all_checks_passed = syntax_passed and satisfiable_passed and deterministic_passed

    return {
        "line_number": record.get("_line_number"),
        "key": record.get("key", ""),
        "task": key_metadata.get("task", ""),
        "representation": key_metadata.get("representation", ""),
        "kg": graph_id,
        "graph_id": graph_id,
        "prompt_type": key_metadata.get("prompt_type", ""),
        "temperature": key_metadata.get("temperature"),
        "temperature_label": key_metadata.get("temperature_label", ""),
        "cq_index": cq_index,
        "cq_complexity": cq_complexity,
        "endpoint": normalize_endpoint(endpoint),
        "competency_question": competency_question,
        "sparql_query": query,
        "syntax_ok": syntax_passed,
        "syntax_message": syntax_message,
        "satisfiable": satisfiable_passed,
        "satisfiable_message": satisfiable_message,
        "deterministic": deterministic_passed,
        "deterministic_message": deterministic_message,
        "all_checks_passed": all_checks_passed,
        "rows": rows,
        "variables": variables,
        "result_message": result_message,
    }


def _as_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if isinstance(value, str):
        return value.strip().lower() in {"true", "1", "yes"}
    return bool(value)


def _row_all_checks_passed(row: Dict[str, Any]) -> bool:
    if "all_checks_passed" in row:
        return _as_bool(row.get("all_checks_passed"))
    return (
        _as_bool(row.get("syntax_ok"))
        and _as_bool(row.get("satisfiable"))
        and _as_bool(row.get("deterministic"))
    )


def _mean(values: List[float]) -> Optional[float]:
    clean = [float(value) for value in values if value is not None]
    if not clean:
        return None
    return round(sum(clean) / len(clean), 4)


def _group_summary(rows: List[Dict[str, Any]], dimensions: List[str], level: str) -> List[Dict[str, Any]]:
    grouped: Dict[Tuple[Any, ...], List[Dict[str, Any]]] = {}
    for row in rows:
        key = tuple(row.get(dimension, "") for dimension in dimensions)
        grouped.setdefault(key, []).append(row)

    summaries = []
    for group_key, group_rows in sorted(grouped.items(), key=lambda item: tuple(str(x) for x in item[0])):
        total = len(group_rows)
        syntax_count = sum(_as_bool(row.get("syntax_ok")) for row in group_rows)
        satisfiable_count = sum(_as_bool(row.get("satisfiable")) for row in group_rows)
        deterministic_count = sum(_as_bool(row.get("deterministic")) for row in group_rows)
        all_checks_count = sum(_row_all_checks_passed(row) for row in group_rows)
        nonzero_rows_count = sum((row.get("rows") or 0) > 0 for row in group_rows)
        summary = {
            "level": level,
            "dimensions": "|".join(dimensions),
            "n": total,
            "syntax_ok_count": syntax_count,
            "syntax_ok_rate": round(syntax_count / total, 4) if total else 0,
            "satisfiable_count": satisfiable_count,
            "satisfiable_rate": round(satisfiable_count / total, 4) if total else 0,
            "deterministic_count": deterministic_count,
            "deterministic_rate": round(deterministic_count / total, 4) if total else 0,
            "all_checks_passed_count": all_checks_count,
            "all_checks_passed_rate": round(all_checks_count / total, 4) if total else 0,
            "nonzero_rows_count": nonzero_rows_count,
            "nonzero_rows_rate": round(nonzero_rows_count / total, 4) if total else 0,
            "avg_rows": _mean([row.get("rows") for row in group_rows]),
        }
        summary.update(dict(zip(dimensions, group_key)))
        summaries.append(summary)
    return summaries


def aggregate_evaluation_results(
    input_jsonl: str = DEFAULT_OUTPUT_JSONL,
    output_jsonl: Optional[str] = DEFAULT_SUMMARY_JSONL,
    output_csv: Optional[str] = DEFAULT_SUMMARY_CSV,
) -> List[Dict[str, Any]]:
    """
    Aggregate row-level evaluation results from broad to detailed effects.

    Analysis variables:
      - kg
      - cq_complexity
      - representation
      - prompt_type
      - temperature_label

    The output includes every non-empty combination of those variables, from
    individual effects through the full five-way interaction.
    """
    rows = list(iter_jsonl(input_jsonl))
    analysis_dimensions = ["kg", "cq_complexity", "representation", "prompt_type", "temperature_label"]
    levels = [
        ("__".join(dimensions), list(dimensions))
        for size in range(1, len(analysis_dimensions) + 1)
        for dimensions in combinations(analysis_dimensions, size)
    ]

    summaries: List[Dict[str, Any]] = []
    for level, dimensions in levels:
        summaries.extend(_group_summary(rows, dimensions, level))

    if output_jsonl:
        with open(output_jsonl, "w", encoding="utf-8") as handle:
            for summary in summaries:
                handle.write(json.dumps(summary, ensure_ascii=False) + "\n")
        print(f"Saved summary JSONL: {output_jsonl}")

    if output_csv and summaries:
        fieldnames = sorted({field for summary in summaries for field in summary})
        preferred = [
            "level", "dimensions", "kg", "representation", "prompt_type", "temperature_label", "cq_complexity",
            "n", "syntax_ok_count", "syntax_ok_rate", "satisfiable_count", "satisfiable_rate",
            "deterministic_count", "deterministic_rate", "all_checks_passed_count", "all_checks_passed_rate",
            "nonzero_rows_count", "nonzero_rows_rate",
            "avg_rows",
        ]
        fieldnames = preferred + [field for field in fieldnames if field not in preferred]
        with open(output_csv, "w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(summaries)
        print(f"Saved summary CSV: {output_csv}")

    return summaries


def _format_rate(value: Any) -> str:
    if value is None:
        return "n/a"
    return f"{float(value) * 100:.1f}%"


def _summary_label(row: Dict[str, Any]) -> str:
    dimensions = [d for d in str(row.get("dimensions", "")).split("|") if d]
    parts = []
    for dimension in dimensions:
        value = row.get(dimension, "")
        if value not in (None, ""):
            parts.append(f"{dimension}={value}")
    return ", ".join(parts) if parts else row.get("level", "overall")


def print_summary_table(
    summaries: List[Dict[str, Any]],
    level: str,
    title: str,
    limit: Optional[int] = None,
) -> None:
    rows = [row for row in summaries if row.get("level") == level]
    if not rows:
        return
    if limit is not None:
        rows = rows[:limit]

    print(f"\n{title}")
    print("-" * len(title))
    for row in rows:
        print(
            f"{_summary_label(row)} | "
            f"n={row.get('n', 0)} | "
            f"syntax={_format_rate(row.get('syntax_ok_rate'))} | "
            f"satisfiable={_format_rate(row.get('satisfiable_rate'))} | "
            f"deterministic={_format_rate(row.get('deterministic_rate'))} | "
            f"all_passed={_format_rate(row.get('all_checks_passed_rate'))} | "
            f"nonzero_rows={_format_rate(row.get('nonzero_rows_rate'))} | "
            f"avg_rows={row.get('avg_rows')}"
        )


def print_final_stats(
    row_results_jsonl: str,
    summaries: List[Dict[str, Any]],
    generated_files: List[Optional[str]],
) -> None:
    """Print compact final stats alongside generated output files."""
    rows = list(iter_jsonl(row_results_jsonl)) if os.path.exists(row_results_jsonl) else []
    print("\nGenerated files")
    print("---------------")
    for file_path in generated_files:
        if file_path:
            print(file_path)

    print("\nOverall evaluation")
    print("------------------")
    if not rows:
        print("No evaluated rows found.")
        return

    total = len(rows)
    syntax_count = sum(_as_bool(row.get("syntax_ok")) for row in rows)
    satisfiable_count = sum(_as_bool(row.get("satisfiable")) for row in rows)
    deterministic_count = sum(_as_bool(row.get("deterministic")) for row in rows)
    all_checks_count = sum(_row_all_checks_passed(row) for row in rows)
    nonzero_rows_count = sum((row.get("rows") or 0) > 0 for row in rows)
    unknown_complexity_count = sum(row.get("cq_complexity") == "unknown" for row in rows)
    print(f"rows={total}")
    print(f"syntax_ok={syntax_count}/{total} ({_format_rate(syntax_count / total)})")
    print(f"satisfiable={satisfiable_count}/{total} ({_format_rate(satisfiable_count / total)})")
    print(f"deterministic={deterministic_count}/{total} ({_format_rate(deterministic_count / total)})")
    print(f"all_checks_passed={all_checks_count}/{total} ({_format_rate(all_checks_count / total)})")
    print(f"nonzero_rows={nonzero_rows_count}/{total} ({_format_rate(nonzero_rows_count / total)})")
    print(f"unknown_cq_complexity={unknown_complexity_count}/{total} ({_format_rate(unknown_complexity_count / total)})")

    print_summary_table(summaries, "kg", "By KG")
    print_summary_table(summaries, "cq_complexity", "By CQ Complexity")
    print_summary_table(summaries, "representation", "By Representation")
    print_summary_table(summaries, "prompt_type", "By Prompt Type")
    print_summary_table(summaries, "temperature_label", "By Temperature")
    print_summary_table(summaries, "kg__representation", "By KG And Representation")
    print_summary_table(summaries, "representation__prompt_type", "By Representation And Prompt Type")
    print_summary_table(summaries, "prompt_type__temperature_label", "By Prompt Type And Temperature")
    print_summary_table(
        summaries,
        "kg__cq_complexity__representation__prompt_type__temperature_label",
        "Full Interaction: KG, Complexity, Representation, Prompt, Temperature",
        limit=80,
    )


def evaluate_jsonl(
    input_jsonl: str = DEFAULT_RESULTS_JSONL,
    output_jsonl: str = DEFAULT_OUTPUT_JSONL,
    output_csv: Optional[str] = DEFAULT_OUTPUT_CSV,
    summary_jsonl: Optional[str] = DEFAULT_SUMMARY_JSONL,
    summary_csv: Optional[str] = DEFAULT_SUMMARY_CSV,
    endpoint_by_graph: Optional[Dict[str, str]] = None,
    runs: int = 3,
    timeout: int = 600,
    limit: Optional[int] = None,
    resume: bool = True,
) -> List[Dict[str, Any]]:
    """
    Evaluate generated SPARQLs in a Gemini batch JSONL file.

    Produces row-level results plus grouped summaries for KG/representation/prompt/temperature effects.
    """
    done_keys = set()
    if resume and os.path.exists(output_jsonl):
        for old in iter_jsonl(output_jsonl):
            done_keys.add(old.get("key"))
        if done_keys:
            print(f"Resuming: {len(done_keys)} rows already in {output_jsonl}")

    results = []
    processed = 0
    mode = "a" if resume and os.path.exists(output_jsonl) else "w"
    with open(output_jsonl, mode, encoding="utf-8") as out:
        for record in iter_jsonl(input_jsonl):
            if limit is not None and processed >= limit:
                break
            if resume and record.get("key") in done_keys:
                continue

            result = evaluate_record(record, endpoint_by_graph=endpoint_by_graph, runs=runs, timeout=timeout)
            out.write(json.dumps(result, ensure_ascii=False) + "\n")
            out.flush()
            results.append(result)
            processed += 1
            print(
                f"{record.get('_line_number')}: "
                f"kg={result['kg']} "
                f"rep={result['representation']} "
                f"prompt={result['prompt_type']} "
                f"temp={result['temperature_label']} "
                f"complexity={result['cq_complexity']} "
                f"syntax_ok={result['syntax_ok']} "
                f"satisfiable={result['satisfiable']} "
                f"deterministic={result['deterministic']} "
                f"all_passed={result['all_checks_passed']} "
                f"rows={result['rows']}"
            )

    if output_csv:
        rows = list(iter_jsonl(output_jsonl))
        if rows:
            preferred = [
                "line_number", "key", "task", "representation", "kg", "graph_id",
                "prompt_type", "temperature", "temperature_label", "cq_index", "cq_complexity",
                "endpoint", "competency_question", "sparql_query",
                "syntax_ok", "syntax_message", "satisfiable", "satisfiable_message",
                "deterministic", "deterministic_message", "all_checks_passed",
                "rows", "variables", "result_message",
            ]
            all_fields = {field for row in rows for field in row.keys()}
            fieldnames = preferred + [field for field in sorted(all_fields) if field not in preferred]
            with open(output_csv, "w", encoding="utf-8", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=fieldnames)
                writer.writeheader()
                for row in rows:
                    row = dict(row)
                    row["variables"] = json.dumps(row.get("variables", []), ensure_ascii=False)
                    writer.writerow(row)
            print(f"Saved CSV: {output_csv}")

    summaries = []
    if summary_jsonl or summary_csv:
        summaries = aggregate_evaluation_results(output_jsonl, output_jsonl=summary_jsonl, output_csv=summary_csv)

    print(f"Saved JSONL: {output_jsonl}")
    print_final_stats(output_jsonl, summaries, [output_jsonl, output_csv, summary_jsonl, summary_csv])
    return results


In [ ]:
# Example usage:
#
results = evaluate_jsonl(
    input_jsonl=DEFAULT_RESULTS_JSONL,
    output_jsonl=DEFAULT_OUTPUT_JSONL,
    output_csv=DEFAULT_OUTPUT_CSV,
    summary_jsonl=DEFAULT_SUMMARY_JSONL,
    summary_csv=DEFAULT_SUMMARY_CSV,
    runs=3,
    timeout=600,
    limit=None
)

# To rebuild summaries from an existing row-level evaluation file:
summaries = aggregate_evaluation_results(
    input_jsonl=DEFAULT_OUTPUT_JSONL,
    output_jsonl=DEFAULT_SUMMARY_JSONL,
    output_csv=DEFAULT_SUMMARY_CSV,
)